In [ ]:
import pandas as pd
import spacy
from tqdm import tqdm
from huggingface_hub import hf_hub_download, snapshot_download
import sys
import json
import random

nlp = spacy.load("en_core_web_md")

ckpt_path = hf_hub_download(
    repo_id="yeomtong/srl_bert_model",
    filename="best_srl_Sep_29.ckpt")
repo_dir = snapshot_download("yeomtong/srl_bert_model")
sys.path.append(repo_dir)

from predictor_up import srl_init
from model import PredicateAwareSRL
from visualizer_up import prediction, prediction_formatted

srl_init(ckpt_path, bert_name= "bert-base-cased")

In [ ]:
import pickle
import pandas as pd

def save_pkl(tgt_list, svg_path):
    with open(svg_path, "wb") as f:
        pickle.dump(tgt_list, f)

def load_pkl(path) :
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data



In [ ]:
train_df = load_pkl("/Enter-your-path/all_train.pkl")
# df.head()
dev_df = load_pkl("/Enter-your-path/all_dev.pkl")
test_df = load_pkl("/Enter-your-path/all_test.pkl")

In [ ]:
import re
import unicodedata

def clean_text_for_srl(text):
    if not isinstance(text, str):
        return ""

    # Normalize Unicode
    text = unicodedata.normalize("NFKC", text)
    text = text.encode("utf-8", "ignore").decode("utf-8")
    text = text.replace("Â", "").replace("Ã", "A").replace("â€”", "—").replace("â€“", "-")
    text = text.replace("±", "+/-").replace("°", " degrees ")
    text = re.sub(r"\([^)]{40,}\)", "", text)   # overly long (...)
    text = re.sub(r"http\S+|www\S+", "", text)  # URLs
    text = re.sub(r"\s+", " ", text).strip()

    # Truncate long sentences (SRL model's limit is 150)
    if len(text.split()) > 150:
        text = " ".join(text.split()[:150])

    return text

In [ ]:
# Given word list and BIO tags for one predicate frame, return [pred_idx, arg0_idx, arg1_idx, arg2_idx, argm_idx] 
def extract_arg_heads(words, tags):
    pred_idx = -1
    arg0_idx = -1
    arg1_idx = -1
    arg2_idx = -1
    argm_idx = -1

    for i, tag in enumerate(tags):
        # predicate
        if tag == "B-V" and pred_idx == -1:
            pred_idx = i
        if tag.startswith("B-ARG0") and arg0_idx == -1:
            arg0_idx = i
        if tag.startswith("B-ARG1") and arg1_idx == -1:
            arg1_idx = i
        if tag.startswith("B-ARG2") and arg2_idx == -1:
            arg2_idx = i
        if tag.startswith("B-ARGM") and argm_idx == -1:
            argm_idx = i

    return [pred_idx, arg0_idx, arg1_idx, arg2_idx, argm_idx]


In [ ]:
# Run SRL once per utterance and return one record per sentence, each with the same politeness label.
def srl_info_for_utterance(utterance_text, formality_score):
    utterance_text = clean_text_for_srl(utterance_text)
    if not isinstance(utterance_text, str) or not utterance_text.strip():
        return {"frames": [], "formality": formality_score}
    
    doc = nlp(utterance_text)
    sentences = [sent.text.strip() for sent in doc.sents]
    
    records = {"frames": []}
    
    for sent in sentences:
        if not sent:
            continue
        
        srl_out = prediction_formatted(sent)
        
        words = srl_out.get("words", [])
        verbs = srl_out.get("verbs", [])
        
        for frame in verbs:
            tags = frame.get("tags", [])
            if len(tags) != len(words):
                continue

            heads = extract_arg_heads(words, tags)
            pred_idx = heads[0]
            if pred_idx < 0:
                continue

            rec = {
                "words": words,
                "predicate_word_idx": pred_idx,
                "labels": tags,
                "arg_head_idx": heads,
            }
            records["frames"].append(rec)
    
    records["formality"] = formality_score
    return records


In [ ]:
from tqdm import tqdm

def get_srl_processed(df):
    all_records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building SRL-aware formality samples"):
        utterance_text = row["sent"]
        formality_score = row["label"]
        records = srl_info_for_utterance(utterance_text, formality_score)
        all_records.append(records)
        
    return all_records

train_records = get_srl_processed(train_df)
dev_records = get_srl_processed(dev_df)
test_records = get_srl_processed(test_df)

In [ ]:
def get_srl_processed(df):
    all_records = []
    skipped = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building SRL-aware formality samples"):
        utterance_text = row["sent"]
        formality_score = row["label"]

        try:
            records = srl_info_for_utterance(utterance_text, formality_score)
            all_records.append(records)
        except Exception as e:
            skipped.append({
                "idx": idx,
                "error": str(e),
                "text": utterance_text
            })
            continue

    print(f"Processed: {len(all_records)}")
    print(f"Skipped: {len(skipped)}")
    return all_records, skipped

train_records = get_srl_processed(train_df)
dev_records = get_srl_processed(dev_df)
test_records = get_srl_processed(test_df)

In [ ]:
save_pkl(train_records,  '/Enter-your-path/train_processed_records.pkl')
save_pkl(dev_records,  '/Enter-your-path/dev_processed_records.pkl')
save_pkl(test_records,  '/Enter-your-path/test_processed_records.pkl')


In [ ]:
import json

def save_records_to_jsonl(records, out_path):
    if isinstance(records, tuple):
        records = records[0]

    with open(out_path, "w", encoding="utf-8") as f:
        for ex in records:
            if ex is None or not isinstance(ex, dict):
                continue

            frames = ex.get("frames", [])

            # pull utterance-level words from the first frame
            top_words = []
            if len(frames) > 0 and isinstance(frames[0], dict):
                top_words = frames[0].get("words", [])

            out = {
                "words": top_words,
                "frames": [
                    {
                        "predicate_word_idx": fr["predicate_word_idx"],
                        "labels": fr["labels"],
                        "predicate_form": fr.get("predicate_form", None),
                        "arg_head_idx": fr.get("arg_head_idx", None),
                    }
                    for fr in frames
                ],
                "formality": ex.get("formality", None),
            }

            f.write(json.dumps(out) + "\n")

In [ ]:
save_records_to_jsonl(train_records, '/Enter-your-path/train_formality.alinged.jsonl')
save_records_to_jsonl(dev_records, '/Enter-your-path/dev_formality.alinged.jsonl')
save_records_to_jsonl(test_records, '/Enter-your-path/test_formality.alinged.jsonl')


In [ ]:
import json

def save_records_to_aligned_jsonl(records, out_path):
    if isinstance(records, tuple):
        records = records[0]

    with open(out_path, "w", encoding="utf-8") as f:
        for ex in records:
            if ex is None or not isinstance(ex, dict):
                continue

            frames = ex.get("frames", [])
            if not frames:
                out = {
                    "words": [],
                    "frames": [],
                    "formality": ex.get("formality", None),
                }
                f.write(json.dumps(out) + "\n")
                continue

            # use utterance words from first frame
            utter_words = frames[0].get("words", [])
            N = len(utter_words)

            aligned_frames = []
            for fr in frames:
                fr_words = fr.get("words", [])
                fr_labels = fr.get("labels", [])

                # Case 1: already aligned
                if len(fr_labels) == N:
                    labels_utt = fr_labels

                # Case 2: sentence-level labels with same tokenization as utterance words
                elif fr_words == utter_words:
                    labels_utt = fr_labels

                # Case 3: cannot safely align
                else:
                    # skip malformed frame
                    continue

                # final safety check
                if len(labels_utt) != N:
                    continue

                aligned_frames.append({
                    "predicate_word_idx": fr["predicate_word_idx"],
                    "labels": labels_utt,
                    "predicate_form": fr.get("predicate_form", None),
                    "arg_head_idx": fr.get("arg_head_idx", None),
                })

            out = {
                "words": utter_words,
                "frames": aligned_frames,
                "formality": ex.get("formality", None),
            }
            f.write(json.dumps(out) + "\n")

In [ ]:
save_records_to_aligned_jsonl(
    train_records,
    "/Enter-your-path/train_formality_2.aligned.jsonl"
)
save_records_to_aligned_jsonl(
    dev_records,
    "/Enter-your-path/dev_formality_2.aligned.jsonl"
)
save_records_to_aligned_jsonl(
    test_records,
    "/Enter-your-path/test_formality_2.aligned.jsonl"
)

In [ ]:
test_records[0][0]